In [1]:
# ============================================================================
# COSADAM NLP ABLATION NOTEBOOK – DistilBERT on AG News
# Researcher: Dr. Ali Zeydi Abdian
# Addresses: R1 (Ablation on c and alpha_cos), R2 (Stats, Seeds), R3 (Large Data)
#
# EXPERT FIXES RETAINED: 
# - Fixed Fatal Statistical Paradox (d_z vs p-value), Cleaned Metrics
# - dynamic subplot generation for Raw Cosine & Smoothed (s) parameters.
# - Auto-ZIP export for Kaggle output, LIVE EPOCH LOGGING
# - UPGRADED TO DistilBERT for ultra-fast, data-efficient ablation
# - INTEGRATED Multi-GPU DataParallel scaling & Upfront Pre-Tokenization
# - UPGRADED VISUALIZATIONS: Added Std-Dev shaded confidence bands
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import os, time, warnings, json, random, zipfile, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, confusion_matrix)
from scipy.stats import ttest_rel, t as t_dist
from statsmodels.stats.multitest import multipletests

# ============================================================================
# 🛡️ ENVIRONMENT & REPRODUCIBILITY
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false" 

warnings.filterwarnings("ignore")
try:
    torch.set_float32_matmul_precision('high')
except:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Device Verified: {device}", flush=True)

ABLATION_SEEDS = [42, 123, 777]  

AG_NEWS_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset'
DISTILBERT_PATH = '/kaggle/input/datasets/alkanerturan/distilbert'

RESULTS_DIR = "/kaggle/working/results_cosadam_nlp_ablation"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "ablation_checkpoint.json")

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================
NLP_EPOCHS = 3
BATCH_SIZE = 128            
MAX_LENGTH = 128
GRADIENT_CLIP_VALUE = 1.0
WARMUP_RATIO = 0.1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 🧠 COSADAM OPTIMIZER (Modified for Tracking)
# ============================================================================
class CosAdam(optim.Optimizer):
    def __init__(self, params, lr=3e-5, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, alpha_cos=0.9, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, alpha_cos=alpha_cos, c=c)
        super(CosAdam, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad'] = torch.zeros_like(p)
                    state['s'] = 0.0  
                    state['raw_cos'] = 0.0

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad, s = state['prev_grad'], state['s']
                beta1, beta2 = group['betas']
                state['step'] += 1

                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    s = group['alpha_cos'] * s + (1 - group['alpha_cos']) * cos_theta.item()
                    state['raw_cos'] = cos_theta.item()
                else:
                    state['raw_cos'] = 0.0
                state['s'] = s

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                step_size = group['lr'] / bias_correction1

                update = -step_size * exp_avg / denom
                update.mul_(1 + group['c'] * s)

                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])

                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss

# ============================================================================
# 📂 DATA LOADER (UPFRONT RAM PRE-TOKENIZATION)
# ============================================================================
def load_agnews_data(tokenizer, max_length=128):
    train_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'train.csv'))
    test_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'test.csv'))
    
    for df in [train_df, test_df]:
        if 'Class Index' in df.columns:
            df.rename(columns={'Class Index': 'label'}, inplace=True)
        if 'Description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'Description': 'text'}, inplace=True)
        elif 'description' in df.columns and 'text' not in df.columns:
            df.rename(columns={'description': 'text'}, inplace=True)
        if 'text' not in df.columns:
            label_col = 'label' if 'label' in df.columns else df.columns[0]
            df['text'] = df[[c for c in df.columns if c != label_col]].astype(str).agg(' '.join, axis=1)
            df.rename(columns={label_col: 'label'}, inplace=True)
        if df['label'].min() == 1:
            df['label'] -= 1

    print("⚡ Pre-tokenizing dataset into RAM for maximum iteration speed...", flush=True)
    train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, padding='max_length', max_length=max_length)
    val_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding='max_length', max_length=max_length)

    class PreTokenizedDataset(Dataset):
        def __init__(self, encodings, labels):
            self.input_ids = encodings['input_ids']
            self.attention_mask = encodings['attention_mask']
            self.labels = labels
        def __len__(self): return len(self.labels)
        def __getitem__(self, idx):
            return {
                'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
                'attention_mask': torch.tensor(self.attention_mask[idx], dtype=torch.long),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)
            }

    train_ds = PreTokenizedDataset(train_encodings, train_df['label'].tolist())
    val_ds = PreTokenizedDataset(val_encodings, test_df['label'].tolist())
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, val_loader

# ============================================================================
# 🔄 TRAINING LOOP
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, scheduler, num_epochs, use_amp=True):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_loss = float('inf')
    best_state = None
    patience = 5
    no_improve = 0

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': []}
    raw_cosine_hist = []
    s_hist = []

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)

            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs.logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.step()

            scheduler.step()
            preds = outputs.logits.argmax(dim=1)
            tr_loss += loss.item() * labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                val_loss += loss.item() * labels.size(0)
                val_correct += outputs.logits.argmax(1).eq(labels).sum().item()
                val_total += labels.size(0)

        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if isinstance(optimizer, CosAdam) or hasattr(optimizer, 'module'):
            opt = optimizer.module if hasattr(optimizer, 'module') else optimizer
            if hasattr(opt, 'param_groups') and opt.param_groups[0]['params']:
                p0 = opt.param_groups[0]['params'][0]
                state = opt.state.get(p0, {})
                raw_cosine_hist.append(state.get('raw_cos', 0.0))
                s_hist.append(state.get('s', 0.0))
        else:
            raw_cosine_hist.append(0.0)
            s_hist.append(0.0)

        print(f"      🔹 Epoch {epoch+1}/{num_epochs} | Val Acc: {history['val_acc'][-1]:.4f} | Time: {epoch_time:.0f}s", flush=True)

        if history['val_loss'][-1] < best_val_loss - 1e-4:
            best_val_loss = history['val_loss'][-1]
            no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            
        if no_improve >= patience and epoch >= patience:
            if best_state:
                model.load_state_dict(best_state)
                model = model.to(device)
            break

    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = F.softmax(outputs.logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(outputs.logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.vstack(all_probs)

    acc = float((all_preds == all_targets).mean())
    f1 = float(f1_score(all_targets, all_preds, average='macro', zero_division=0))
    try: auc_val = float(roc_auc_score(all_targets, all_probs, multi_class='ovr', average='macro'))
    except: auc_val = 0.0
    mcc = float(matthews_corrcoef(all_targets, all_preds))
    
    return {
        'test_acc': acc, 'test_f1': f1, 'test_auc': auc_val, 'test_mcc': mcc,
        'avg_epoch_time': float(np.mean(history['epoch_time'])), 'history': history,
        'raw_cosine_hist': raw_cosine_hist, 's_hist': s_hist
    }

# ============================================================================
# 📊 STATISTICAL ANALYSIS & CHECKPOINTING
# ============================================================================
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f: return json.load(f)
        except json.JSONDecodeError: pass
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w') as f: json.dump(results, f, indent=4)

def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        margin = t_dist.ppf((1 + cl) / 2, max(n - 1, 1)) * std / np.sqrt(n) if n > 0 else 0
        return float(mean - margin), float(mean + margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    alpha = (1 - cl) / 2
    return float(np.percentile(boot, 100 * alpha)), float(np.percentile(boot, 100 * (1 - alpha)))

def compute_statistics(multi_results):
    metrics = ['test_acc', 'test_f1', 'test_auc', 'test_mcc', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs]
            mean_v = float(np.mean(vals))
            std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'] = mean_v
            agg[opt][f'{m}_std'] = std_v
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            cvs[opt][f'{m}_cv'] = float((std_v / mean_v * 100) if mean_v > 0 else 0)
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(results, baseline='CosAdam_c=0.5_α=0.9'):
    if baseline not in results or len(results[baseline]) < 2: return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig, pvals, basenames = {}, [], []
    
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2: continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals) != len(vals): continue
            
        basenames.append(opt)
        diffs = base_vals - vals
        mean_diff, std_diff = float(np.mean(diffs)), float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z, p_raw = 0.0, (1.0 if abs(mean_diff) < 1e-8 else 0.0)
        else:
            d_z = float(mean_diff / std_diff)
            try: _, p_raw = ttest_rel(base_vals, vals)
            except: p_raw = 1.0
                
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_diff, 'p_raw': float(p_raw), 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(basenames):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = '***' if p_corr[i] < 0.001 else '**' if p_corr[i] < 0.01 else '*' if p_corr[i] < 0.05 else 'ns'
    return sig

# ============================================================================
# 📄 VISUALIZATION & LATEX FUNCTIONS
# ============================================================================
def ablation_performance_table(agg, ci, caption, label, filename):
    mc = [{'d':'Acc','k':'test_acc','f':'.4f'}, {'d':'F1','k':'test_f1','f':'.4f'},
          {'d':'AUC','k':'test_auc','f':'.4f'}, {'d':'MCC','k':'test_mcc','f':'.4f'},
          {'d':'Time(s)','k':'avg_epoch_time','f':'.2f'}]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
        r"{\bf Config} & " + " & ".join([f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"
    ]
    order = ['AdamW_Baseline', 'CosAdam_c=0.1_α=0.9', 'CosAdam_c=0.5_α=0.9', 'CosAdam_c=1.0_α=0.9', 
             'CosAdam_c=0.5_α=0.0', 'CosAdam_c=0.5_α=0.99']
    for opt in order:
        if opt not in agg: continue
        row = [opt.replace('_', ' ').replace('α', r'$\alpha$')]
        for m in mc:
            mean, std, (low, high) = agg[opt][f"{m['k']}_mean"], agg[opt][f"{m['k']}_std"], ci[opt][m['k']]
            row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}} \\; [{low:{m['f']}}, {high:{m['f']}}]")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\begin{tabular}{lccccc}", r"\toprule",
        r"\textbf{Comparison vs $c$=0.5, $\alpha$=0.9} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\",
        r"\midrule"
    ]
    for opt, r in sig.items():
        diff_str = f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$"
        lines.append(f"{opt.replace('_',' ').replace('α', r'$\alpha$')} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def variance_table(var, cvs, caption, label, filename):
    metrics_show = [('test_acc', 'Accuracy'), ('test_f1', 'F1'), ('test_auc', 'AUC')]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}", r"\label{" + label + "}",
        r"\begin{tabular}{l" + "cc" * len(metrics_show) + "}", r"\toprule",
        r"\multirow{2}{*}{\bf Config} " + "".join([f"& \\multicolumn{{2}}{{c}}{{\\bf {disp}}}" for _, disp in metrics_show]) + r" \\",
        r"\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7}",
        r" & " + " & ".join([r"$\sigma^2$ & CV\%"] * len(metrics_show)) + r" \\", r"\midrule"
    ]
    for opt in var.keys():
        row = [opt.replace('_', ' ').replace('α', r'$\alpha$')]
        for key, _ in metrics_show:
            v = var[opt][f'{key}_var']
            row.append(r"$<10^{-6}$" if v < 1e-6 else f"{v:.2e}")
            row.append(f"{cvs[opt][f'{key}_cv']:.2f}\\%")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

# ----------------------------------------------------------------------------
# UPGRADED PLOTTING FUNCTIONS: Shaded Confidence Bands (±1 Std Dev)
# ----------------------------------------------------------------------------
def plot_learning_curves(results, dataset_name="DistilBERT on AG News", save_path=None):
    """Plot learning curves for Accuracy and Loss with Shaded Confidence Bands."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10.colors
    line_styles = ['-', '--']

    for idx, (name, res) in enumerate(results.items()):
        if len(res) == 0: continue

        all_train_acc, all_val_acc = [], []
        all_train_loss, all_val_loss = [], []
        max_epochs = max([len(r['history']['train_acc']) for r in res])

        for r in res:
            history = r['history']
            train_acc, val_acc = list(history['train_acc']), list(history['val_acc'])
            train_loss, val_loss = list(history['train_loss']), list(history['val_loss'])

            # Pad with the last value for early stopped runs
            if len(train_acc) < max_epochs:
                train_acc += [train_acc[-1]] * (max_epochs - len(train_acc))
                val_acc += [val_acc[-1]] * (max_epochs - len(val_acc))
                train_loss += [train_loss[-1]] * (max_epochs - len(train_loss))
                val_loss += [val_loss[-1]] * (max_epochs - len(val_loss))

            all_train_acc.append(train_acc); all_val_acc.append(val_acc)
            all_train_loss.append(train_loss); all_val_loss.append(val_loss)

        epochs = range(1, max_epochs + 1)
        mean_train_acc, std_train_acc = np.mean(all_train_acc, axis=0), np.std(all_train_acc, axis=0)
        mean_val_acc, std_val_acc = np.mean(all_val_acc, axis=0), np.std(all_val_acc, axis=0)
        mean_train_loss, std_train_loss = np.mean(all_train_loss, axis=0), np.std(all_train_loss, axis=0)
        mean_val_loss, std_val_loss = np.mean(all_val_loss, axis=0), np.std(all_val_loss, axis=0)

        clean_name = name.replace('_', ' ').replace('α', r'$\alpha$')

        # Plot Accuracy
        ax1.plot(epochs, mean_train_acc, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
        ax1.plot(epochs, mean_val_acc, label=f'{clean_name}', color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
        ax1.fill_between(epochs, mean_train_acc - std_train_acc, mean_train_acc + std_train_acc, color=colors[idx % len(colors)], alpha=0.1)
        ax1.fill_between(epochs, mean_val_acc - std_val_acc, mean_val_acc + std_val_acc, color=colors[idx % len(colors)], alpha=0.2)

        # Plot Loss
        ax2.plot(epochs, mean_train_loss, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
        ax2.plot(epochs, mean_val_loss, label=f'{clean_name}', color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
        ax2.fill_between(epochs, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, color=colors[idx % len(colors)], alpha=0.1)
        ax2.fill_between(epochs, mean_val_loss - std_val_loss, mean_val_loss + std_val_loss, color=colors[idx % len(colors)], alpha=0.2)

    ax1.set_title(f'{dataset_name} - Accuracy Learning Curves', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.grid(True, alpha=0.3)
    ax1.legend(loc='lower right', fontsize=9)

    ax2.set_title(f'{dataset_name} - Loss Learning Curves', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper right', fontsize=9)

    # Dynamic run count for the text box
    num_runs = len(list(results.values())[0]) if results else 0
    guidance_text = (
        f"Shaded areas represent ±1 standard deviation across {num_runs} independent runs.\n"
        "Solid lines: Training, Dashed lines: Validation\n"
        "Smaller shaded areas indicate more stable and reproducible training."
    )
    plt.figtext(0.5, 0.01, guidance_text, ha='center', fontsize=11, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    
    if not save_path: save_path = os.path.join(RESULTS_DIR, "plots", "learning_curves_acc_loss.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_ablation_dynamics(results, title, save_name):
    """Plot internal optimizer dynamics (Cosine Similarities) with Shaded Confidence Bands."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10.colors
    configs = [c for c in results.keys() if 'AdamW' not in c]
    
    for i, cfg in enumerate(configs):
        runs = results[cfg]
        if not runs: continue
        
        max_epochs = max([len(r['s_hist']) for r in runs])
        all_s, all_raw = [], []
        
        for r in runs:
            s_hist, raw_hist = list(r['s_hist']), list(r['raw_cosine_hist'])
            if len(s_hist) < max_epochs:
                s_hist += [s_hist[-1]] * (max_epochs - len(s_hist))
                raw_hist += [raw_hist[-1]] * (max_epochs - len(raw_hist))
            all_s.append(s_hist); all_raw.append(raw_hist)
            
        epochs = range(1, max_epochs + 1)
        mean_s, std_s = np.mean(all_s, axis=0), np.std(all_s, axis=0)
        mean_raw, std_raw = np.mean(all_raw, axis=0), np.std(all_raw, axis=0)
        clean_name = cfg.replace('_', ' ').replace('α', r'$\alpha$')
        
        # Plot Smoothed Cosine (s)
        ax1.plot(epochs, mean_s, label=clean_name, color=colors[i % len(colors)], linewidth=2)
        ax1.fill_between(epochs, mean_s - std_s, mean_s + std_s, color=colors[i % len(colors)], alpha=0.2)
        
        # Plot Raw Cosine
        ax2.plot(epochs, mean_raw, label=clean_name, color=colors[i % len(colors)], linewidth=1.5, alpha=0.8)
        ax2.fill_between(epochs, mean_raw - std_raw, mean_raw + std_raw, color=colors[i % len(colors)], alpha=0.15)

    ax1.axhline(0.0, color='black', linestyle='--', alpha=0.5)
    ax1.set_title(r'Smoothed Cosine Similarity ($s$)', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.grid(True, linestyle='--', alpha=0.6); ax1.legend(fontsize=9)

    ax2.axhline(0.0, color='black', linestyle='--', alpha=0.5)
    ax2.set_title(r'Raw Gradient Cosine Similarity ($\cos \theta$)', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, "plots", save_name), dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# 📦 FINAL ZIP EXPORT
# ============================================================================
def create_final_zip():
    zip_path = "/kaggle/working/cosadam_ablation_results.zip"
    base_folder = os.path.basename(RESULTS_DIR)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(CHECKPOINT_FILE):
            zipf.write(CHECKPOINT_FILE, arcname=os.path.join(base_folder, os.path.basename(CHECKPOINT_FILE)))
        for sub_dir in ["plots", "latex_tables"]:
            dir_path = os.path.join(RESULTS_DIR, sub_dir)
            if os.path.isdir(dir_path):
                for root, _, files in os.walk(dir_path):
                    for file in files:
                        zipf.write(os.path.join(root, file), arcname=os.path.join(base_folder, sub_dir, file))
    print(f"\n📦 Final results safely archived at: {zip_path}", flush=True)

# ============================================================================
# 🧪 ABLATION EXPERIMENTS WITH SMART CHECKPOINTING
# ============================================================================
def run_ablation_study(tokenizer):
    print("\n=== COSADAM NLP ABLATION STUDY ===")
    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH)
    
    configs = {
        'AdamW_Baseline': lambda p: optim.AdamW(p, lr=3e-5, weight_decay=0.01),
        'CosAdam_c=0.1_α=0.9': lambda p: CosAdam(p, lr=3e-5, weight_decay=0.01, c=0.1, alpha_cos=0.9),
        'CosAdam_c=0.5_α=0.9': lambda p: CosAdam(p, lr=3e-5, weight_decay=0.01, c=0.5, alpha_cos=0.9),
        'CosAdam_c=1.0_α=0.9': lambda p: CosAdam(p, lr=3e-5, weight_decay=0.01, c=1.0, alpha_cos=0.9),
        'CosAdam_c=0.5_α=0.0': lambda p: CosAdam(p, lr=3e-5, weight_decay=0.01, c=0.5, alpha_cos=0.0),
        'CosAdam_c=0.5_α=0.99': lambda p: CosAdam(p, lr=3e-5, weight_decay=0.01, c=0.5, alpha_cos=0.99)
    }
    
    results = load_checkpoint()
    for name in configs:
        if name not in results: results[name] = []

    total_steps = NLP_EPOCHS * len(train_loader)

    for seed in ABLATION_SEEDS:
        for cfg_name, cfg_fn in configs.items():
            if len([r for r in results[cfg_name] if r.get('seed') == seed]) > 0:
                print(f"Skipping {cfg_name} seed={seed} (Already completed)", flush=True)
                continue

            set_seed(seed)
            print(f"\n🔥 Running {cfg_name} with seed={seed}...", flush=True)
            
            model = DistilBertForSequenceClassification.from_pretrained(
                DISTILBERT_PATH, num_labels=4, local_files_only=True, ignore_mismatched_sizes=True
            ).to(device)
            
            if torch.cuda.device_count() > 1:
                print(f"🚀 Utilizing {torch.cuda.device_count()} GPUs with DataParallel!", flush=True)
                model = nn.DataParallel(model)
            
            optimizer = cfg_fn(model.parameters())
            scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
            
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, NLP_EPOCHS)
            res['seed'] = seed
            print(f"   ✔️ {cfg_name} seed={seed} Final Acc={res['test_acc']:.4f}", flush=True)
            
            results[cfg_name].append(res)
            save_checkpoint(results)
            
    return results

# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("=" * 80)
    print("COSADAM NLP ABLATION NOTEBOOK – Statistically Robust & Optimized")
    print("=" * 80)

    tokenizer = DistilBertTokenizerFast.from_pretrained(DISTILBERT_PATH, local_files_only=True)
    ablation_results = run_ablation_study(tokenizer)
    
    abl_agg, abl_var, abl_cvs, abl_ci = compute_statistics(ablation_results)
    ablation_performance_table(abl_agg, abl_ci, r"Ablation Study ($c$ and $\alpha_{cos}$) - DistilBERT on AG News",
                                "tab:nlp_ablation_perf", "t1_ablation_perf.tex")
    
    abl_sig = statistical_tests(ablation_results, baseline='CosAdam_c=0.5_α=0.9')
    stats_table(abl_sig, r"Statistical Significance in Ablation (vs $c$=0.5, $\alpha$=0.9, Paired T-Test) - DistilBERT on AG News",
                "tab:nlp_ablation_stats", "t2_ablation_stats.tex")
                
    variance_table(abl_var, abl_cvs, r"Stability Analysis (Variance \& CV) - DistilBERT on AG News",
                   "tab:nlp_ablation_var", "t3_ablation_var.tex")
    
    # NEW UPGRADED PLOTS
    plot_learning_curves(ablation_results, dataset_name="DistilBERT on AG News")
    plot_ablation_dynamics(ablation_results, "CosAdam Ablation Dynamics (DistilBERT on AG News)", "ablation_dynamics.png")

    create_final_zip()
    print("\n✅ NLP ABLATION COMPLETE. Checkpoints, Results and LaTeX tables saved.")

🚀 Execution Device Verified: cuda
COSADAM NLP ABLATION NOTEBOOK – Statistically Robust & Optimized

=== COSADAM NLP ABLATION STUDY ===
⚡ Pre-tokenizing dataset into RAM for maximum iteration speed...

🔥 Running AdamW_Baseline with seed=42...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9293 | Time: 219s
      🔹 Epoch 2/3 | Val Acc: 0.9366 | Time: 225s
      🔹 Epoch 3/3 | Val Acc: 0.9366 | Time: 225s
   ✔️ AdamW_Baseline seed=42 Final Acc=0.9366

🔥 Running CosAdam_c=0.1_α=0.9 with seed=42...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9320 | Time: 263s
      🔹 Epoch 2/3 | Val Acc: 0.9359 | Time: 262s
      🔹 Epoch 3/3 | Val Acc: 0.9382 | Time: 263s
   ✔️ CosAdam_c=0.5_α=0.0 seed=777 Final Acc=0.9382

🔥 Running CosAdam_c=0.5_α=0.99 with seed=777...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/alkanerturan/distilbert
Key               | Status   |                                                                                     
------------------+----------+-------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([4])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([4, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


🚀 Utilizing 2 GPUs with DataParallel!
      🔹 Epoch 1/3 | Val Acc: 0.9297 | Time: 262s
      🔹 Epoch 2/3 | Val Acc: 0.9362 | Time: 262s
      🔹 Epoch 3/3 | Val Acc: 0.9384 | Time: 262s
   ✔️ CosAdam_c=0.5_α=0.99 seed=777 Final Acc=0.9384

📦 Final results safely archived at: /kaggle/working/cosadam_ablation_results.zip

✅ NLP ABLATION COMPLETE. Checkpoints, Results and LaTeX tables saved.
